# 02 - Silver to Gold (Batch Layer)
Agregations metier + ecriture dans ClickHouse (base OLAP colonnaire)

Les resultats sont stockes dans **HDFS** (Gold) et dans **ClickHouse** pour le dashboard

In [1]:
import subprocess
subprocess.check_call(['pip', 'install', 'clickhouse-connect'])

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import clickhouse_connect

spark = SparkSession.builder \
    .appName("02_Silver_to_Gold") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.sql.shuffle.partitions", "6") \
    .getOrCreate()

# Connexion ClickHouse
ch_client = clickhouse_connect.get_client(host='clickhouse', port=8123)

print("Spark connecte")
print("ClickHouse connecte")

Spark connecte
ClickHouse connecte


In [2]:
# Lire les donnees Silver
trips = spark.read.parquet("hdfs://namenode:9000/projet/silver/trips_clean/")
dates = spark.read.parquet("hdfs://namenode:9000/projet/silver/dates_clean/")
locations = spark.read.parquet("hdfs://namenode:9000/projet/silver/locations_clean/")
weather = spark.read.parquet("hdfs://namenode:9000/projet/silver/weather_clean/")

# Distribuer sur les 2 workers (6 partitions = 3 cores x 2 workers)
trips = trips.repartition(6)

print(f"Trips Silver: {trips.count()} lignes")
print(f"Partitions: {trips.rdd.getNumPartitions()}")

Trips Silver: 894986 lignes
Partitions: 6


In [3]:
# Agregation 1 : Stats par borough
stats_borough = (trips
    .join(broadcast(locations),
          trips.pickup_location_id == locations.location_id)
    .groupBy("borough")
    .agg(
        count("*").alias("nb_trips"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("total_amount"), 2).alias("avg_fare"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy(desc("nb_trips"))
)
stats_borough.show()

# Sauvegarder HDFS + ClickHouse
stats_borough.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_par_borough/")

pdf = stats_borough.toPandas()
ch_client.command("TRUNCATE TABLE batch_stats_borough")
ch_client.insert_df("batch_stats_borough", pdf)
print(f"ClickHouse: {len(pdf)} lignes inserees dans batch_stats_borough")

+-------------+--------+------------+--------+-------------+
|      borough|nb_trips|avg_distance|avg_fare|total_revenue|
+-------------+--------+------------+--------+-------------+
|    Manhattan|  438315|        4.11|    23.9|1.047409077E7|
|        Bronx|  196251|        3.71|   30.48|   5981347.29|
|       Queens|  155238|        8.76|   41.19|   6394851.91|
|     Brooklyn|   94813|        7.05|   25.77|   2443121.37|
|Staten Island|   10341|        3.15|   27.56|    285019.16|
|          EWR|      28|        5.39|  105.59|      2956.42|
+-------------+--------+------------+--------+-------------+

ClickHouse: 6 lignes inserees dans batch_stats_borough


In [4]:
# Agregation 2 : Stats par mois
stats_mensuel = (trips
    .join(broadcast(dates), "date_id")
    .groupBy("month", "year")
    .agg(
        count("*").alias("nb_trips"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("trip_distance"), 2).alias("avg_distance")
    )
    .orderBy("year", "month")
)
stats_mensuel.show(12)

# Sauvegarder HDFS + ClickHouse
stats_mensuel.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_par_mois/")

pdf = stats_mensuel.toPandas()
ch_client.command("TRUNCATE TABLE batch_stats_monthly")
ch_client.insert_df("batch_stats_monthly", pdf)
print(f"ClickHouse: {len(pdf)} lignes inserees dans batch_stats_monthly")

+-----+----+--------+-------------+------------+
|month|year|nb_trips|total_revenue|avg_distance|
+-----+----+--------+-------------+------------+
|    1|2024|   71131|   1946230.13|        3.77|
|    2|2024|   71928|   1960845.74|        3.22|
|    3|2024|   85046|   2376723.92|        3.38|
|    4|2024|   84724|   2388945.63|       10.27|
|    5|2024|   89279|   2586716.69|        6.91|
|    6|2024|   84509|   2432403.95|        5.55|
|    7|2024|   73441|    2134096.6|        3.62|
|    8|2024|   70798|   2061388.09|        5.39|
|    9|2024|   86004|   2531108.72|        5.65|
|   10|2024|   91151|   2682235.81|        4.07|
|   11|2024|   86975|   2480691.64|        3.97|
+-----+----+--------+-------------+------------+

ClickHouse: 11 lignes inserees dans batch_stats_monthly


In [5]:
# Agregation 3 : Top 10 zones
top_zones = (trips
    .join(broadcast(locations),
          trips.pickup_location_id == locations.location_id)
    .groupBy("zone", "borough")
    .agg(count("*").alias("nb_trips"))
    .orderBy(desc("nb_trips"))
    .limit(10)
)
top_zones.show(truncate=False)

# Sauvegarder HDFS + ClickHouse
top_zones.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/top_zones/")

pdf = top_zones.toPandas()
ch_client.command("TRUNCATE TABLE batch_top_zones")
ch_client.insert_df("batch_top_zones", pdf)
print(f"ClickHouse: {len(pdf)} lignes inserees dans batch_top_zones")

+---------------------------------+---------+--------+
|zone                             |borough  |nb_trips|
+---------------------------------+---------+--------+
|Jamaica                          |Queens   |42287   |
|Melrose South                    |Bronx    |42252   |
|University Heights/Morris Heights|Bronx    |41695   |
|Union Sq                         |Manhattan|37697   |
|Middle Village                   |Queens   |31148   |
|Sunset Park West                 |Brooklyn |30106   |
|Pelham Bay Park                  |Bronx    |29668   |
|Lenox Hill East                  |Manhattan|28888   |
|Kingsbridge Heights              |Bronx    |28433   |
|Mott Haven/Port Morris           |Bronx    |25731   |
+---------------------------------+---------+--------+

ClickHouse: 10 lignes inserees dans batch_top_zones


In [6]:
# Agregation 4 : Impact meteo
stats_meteo = (trips
    .filter(col("weather_id") != -1)
    .join(broadcast(weather), "weather_id")
    .groupBy("weather_id", "temp_max", "precipitation", "snow")
    .agg(
        count("*").alias("nb_trips"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("total_amount"), 2).alias("avg_fare")
    )
)
stats_meteo.show()

# Sauvegarder HDFS + ClickHouse
stats_meteo.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_meteo/")

pdf = stats_meteo.toPandas()
pdf['weather_id'] = pdf['weather_id'].astype(int)
ch_client.command("TRUNCATE TABLE batch_stats_weather")
ch_client.insert_df("batch_stats_weather", pdf)
print(f"ClickHouse: {len(pdf)} lignes inserees dans batch_stats_weather")

+----------+--------+-------------+----+--------+------------+--------+
|weather_id|temp_max|precipitation|snow|nb_trips|avg_distance|avg_fare|
+----------+--------+-------------+----+--------+------------+--------+
|      15.0|   24.58|          0.0| 0.0|    3276|        3.43|   29.46|
|      11.0|   10.17|        12.67| 0.0|    3151|        3.23|   28.12|
|       1.0|    7.06|         0.16| 0.0|    1889|        4.38|   31.64|
|      35.0|   14.17|          0.0| 0.0|    2902|        3.51|   28.37|
|      13.0|     5.0|         41.7| 0.0|    3265|        3.39|   28.59|
|       5.0|    6.84|         4.31| 0.0|    2542|        2.99|   26.87|
|      14.0|   21.01|         1.25| 0.0|    2823|        3.27|   28.58|
|      22.0|   28.26|          0.0| 0.0|    1993|        3.61|   28.41|
|      25.0|   33.26|        17.97| 0.0|    2747|        3.28|   26.95|
|      30.0|   21.45|         0.02| 0.0|    2876|        3.24|   28.62|
|       3.0|    5.87|          0.0| 0.0|    1961|         3.9|  

In [7]:
# Agregation 5 : Matrice Origine-Destination (Top 20)
loc_pu = locations.withColumnRenamed("zone", "zone_pickup") \
    .withColumnRenamed("location_id", "pickup_location_id") \
    .select("pickup_location_id", "zone_pickup")

loc_do = locations.withColumnRenamed("zone", "zone_dropoff") \
    .withColumnRenamed("location_id", "dropoff_location_id") \
    .select("dropoff_location_id", "zone_dropoff")

od_matrix = (trips
    .join(broadcast(loc_pu), on="pickup_location_id")
    .join(broadcast(loc_do), on="dropoff_location_id")
    .groupBy("zone_pickup", "zone_dropoff")
    .agg(
        count("*").alias("nb_trips"),
        round(avg("total_amount"), 2).alias("avg_fare")
    )
    .orderBy(desc("nb_trips"))
    .limit(20)
)
od_matrix.show(truncate=False)

# Sauvegarder HDFS + ClickHouse
od_matrix.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/od_matrix/")

pdf = od_matrix.toPandas()
ch_client.command("TRUNCATE TABLE batch_od_matrix")
ch_client.insert_df("batch_od_matrix", pdf)
print(f"ClickHouse: {len(pdf)} lignes inserees dans batch_od_matrix")

print("\nSilver -> Gold TERMINE (HDFS + ClickHouse)")

+---------------------------------+---------------------------------+--------+--------+
|zone_pickup                      |zone_dropoff                     |nb_trips|avg_fare|
+---------------------------------+---------------------------------+--------+--------+
|University Heights/Morris Heights|Union Sq                         |6076    |15.69   |
|Union Sq                         |University Heights/Morris Heights|5255    |16.08   |
|University Heights/Morris Heights|University Heights/Morris Heights|4125    |13.88   |
|Union Sq                         |Union Sq                         |4028    |13.23   |
|Melrose South                    |University Heights/Morris Heights|2920    |17.46   |
|University Heights/Morris Heights|Melrose South                    |2540    |17.71   |
|Melrose South                    |Union Sq                         |2445    |22.15   |
|Upper East Side South            |Lenox Hill East                  |2372    |14.54   |
|Lenox Hill East                

In [8]:
ch_client.close()
spark.stop()
print("Connexions fermees")

Connexions fermees
